In [573]:
#from google.colab import drive # remove the cell if not using colab
#drive.mount('/content/drive')

In [574]:
import pandas as pd
from pathlib import Path
#base_path = Path('/content/drive/Shareddrives/KN Solvro/03. Projekty/Wakacyjne Wyzwanie 20256/ML/notebooki/1_Przetwarzanie i wizualizacja danych') # change path here!
base_path = Path('.') # change path here!

# Klasyfikacja pasażerów Titanica
Nie wiemy czy dla DiCaprio było miejsce na drzwiach, ale wiemy że grdyby był tam Wojfer87 to by z nimi wyciskał pompki na górze lodowej. Teraz twoja pora na wyciskanie.
#Twoje zadnie to:
**stworzenie modelu przewidującego szanse przeżycia katastrofy Titanica**.

![https://i1.jbzd.com.pl/contents/2025/11/normal/v5Fth4DcPpPPxSrXQ5rCbAgZ8EifWiiF.png](https://i1.jbzd.com.pl/contents/2025/11/normal/v5Fth4DcPpPPxSrXQ5rCbAgZ8EifWiiF.png "Wojfer")



#### Twoim celem będzie jest wytrenowanie modeli do klasyfikacji każdego pasażera Titanica jako ofiary (0) lub osoby, która przeżyła (1).

Poniżej znajdziesz pytania, które mogą być pomocne w zadaniu:

- Czego nauczyło Cię o badanym zbiorze danych poprzednie zadanie? Jak możesz wykorzystać wyciągnięte z niego wnioski w procesie tworzenia modelu?
- Jak przeprowadzenie standaryzacji danych może wpływać na zachowanie modelu?
- Co mój model robi i w jaki sposób?
- Jak nie przetrenować wybranego modelu?
- Jaki wynik klasyfikacji możemy uznać za *dobry*?


Wymagania:
- Wypisz obserwacje z pierwszego zadania, które pomogą Ci w tym. Co było przydatne, a co okazało się bezużyteczne?
- [Nie doprowadź](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) do ~~przecieku statku~~ wycieku danych (np. nie ucz modelu na danych testowych). Nauczone modele odpal na danych treningowych i testowych - opisz uzyskane wyniki.
- Stwórz baseline, czyli dla porównania sprawdź jak z zadaniem radzi sobie [Dummy Classifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) (jeśli Twój docelowy model radzi sobie gorzej - uciekaj)
- Przeprowadź badania na dwóch wybranych modelach uczenia maszynowego (np. spośród: drzew decyzyjnych, SVM, MLP, KNN, z gwiazdką [XGBoost](https://xgboost.readthedocs.io/en/stable))
- W badaniach użyj wybranych metryk. Wybór uzasadnij.
- Dla każdego modelu wybierz co najmniej dwa hiperparametry i przeprowadź badania zależności wyników metryk od wartości hiperparametrów. Zwizualizuj wszystko ładnie, zastanów się dlaczego tak mogło być i wyciągnij i wypisz wnioski.
- Podsumuj przeprowadzone badania, wypisz wnioski.

Niezmiennie, zadbaj o czytelność kodu i nazewnictwo zmiennych. Jeśli jakiś wycinek kodu się powtarza, to wyodrębnij go do funkcji. Postaraj się zamieszczać swoje wnioski w postaci komentarza `Markdown`.

Jeśli chcesz, możesz sprawdzić (przyjmując pewne założenia), jakie byłyby Twoje szanse na Titanicu.

Uwaga! Jeśli Titanic to dla Ciebie nic i baaaaardzo chcesz to możesz w ramach tego zadania zająć się [bardziej wymagającym](https://archive.ics.uci.edu/dataset/365/polish+companies+bankruptcy+data) zbiorem.

In [575]:
titanic_df = pd.read_csv(base_path / 'titanic_prepared.csv', index_col='PassengerId')

In [576]:
titanic_df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,HasCabin,IsChild,FamilySize,IsAlone,TicketSize,Embarked_C,Embarked_Q,Embarked_S,Embarked_U
PassengerId,,,,,,,,,,,,,,,,
1,0,3,0,22.0,1,0,7.2500,0,0,2,0,1,0,0,1,0
2,1,1,1,38.0,1,0,71.2833,1,0,2,0,1,1,0,0,0
3,1,3,1,26.0,0,0,7.9250,0,0,1,1,1,0,0,1,0
4,1,1,1,35.0,1,0,53.1000,1,0,2,0,2,0,0,1,0
5,0,3,0,35.0,0,0,8.0500,0,0,1,1,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
887,0,2,0,27.0,0,0,13.0000,0,0,1,1,1,0,0,1,0
888,1,1,1,19.0,0,0,30.0000,1,0,1,1,1,0,0,1,0
889,0,3,1,24.0,1,2,23.4500,0,0,4,0,2,0,0,1,0


## Podsumowanie pierwszego zadania

> Podczas analizy danych udało się zauważyć, że pola `Name`, `Cabin` i `Ticket` warto usunąć, ponieważ po wyciągnięciu z nich danych `HasCabin`, `FamilySize`, `IsAlone`, `TicketSize` już nie dają nam więcej przydatnej informacji dla modelu. 

> Natomiast z macierzy korelacji widzimy silną korelację dla par (`FamilySize`, `IsAlone`), (`FamilySize`, `Parch`), (`FamilySize`, `SibSp`). I też widzimy, że ostatnie z tych dwóch najprawdopodobniej pojawiają się, ponieważ `FamilySize`, `Parch`, `SibSp` razem opisują liczbę osób w rodzinie. Zatem, aby zapobiec przeuczeniu modelu i redundancji, warto usunąć `FamilySize`.


In [577]:
titanic_df.drop("FamilySize", axis=1, inplace=True)

## Podział danych na testowe i treningowe

In [578]:
from sklearn.model_selection import train_test_split

X = titanic_df.drop("Survived", axis=1)
y = titanic_df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, shuffle=True, random_state=5)


## Wybór metryk

W przypadku odpowiedzi `przeżył(1) / nie przeżył(0)` nie wystarczy zwykłej dokładności (accuracy), ponieważ ważna jest różnica pomiędzy odpowiedziami. Lepiej jest powiedzieć, że pasażer nie przeżył niż dawać nieprawdziwą nadzieję. 

> Najlepiej w tym przypadku porównywać precyzję (precision), która odpowiada na pytanie: **Gdy model mówi, że ktoś przeżył, jak często ma rację?**

$$
Precision = \frac{TP}{TP + FP}
$$

Powinniśmy dążyć do tego, żeby ta metryka była jaknajbliżej 1.

## Dummy Classifier

In [579]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, classification_report

dummy_model = DummyClassifier(strategy = 'uniform')
dummy_model.fit(X_train, y_train)


def evaluate_model(y_test, model):
    y_pred = model.predict(X_test)

    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

evaluate_model(y_test, dummy_model)

              precision    recall  f1-score   support

           0       0.65      0.59      0.62       111
           1       0.42      0.47      0.44        68

    accuracy                           0.55       179
   macro avg       0.53      0.53      0.53       179
weighted avg       0.56      0.55      0.55       179

[[66 45]
 [36 32]]


> Możemy zauważyć, że przecyzja modelu opartym na DummyClassifier jest bardzo niska, co oznacza, że model ten nie radzi sobie z tym zadaniem. Wartość precyzji wynosi 0.39, co oznacza, że gdy model mówi, że ktoś przeżył, to tylko w 39% przypadków ma rację.

>Czyli musimy znaleźć lepszy model, który będzie w stanie lepiej przewidzieć szanse przeżycia katastrofy Titanica.

## Drzewa decyzyjne

Spróbujemy użyć drzewa decyzyjnego o wybranej głębokości 3

In [580]:
from sklearn.tree import DecisionTreeClassifier

tree_model = DecisionTreeClassifier(max_depth=3, random_state=5)
tree_model.fit(X_train, y_train)

evaluate_model(y_test, tree_model)

              precision    recall  f1-score   support

           0       0.83      0.90      0.87       111
           1       0.81      0.71      0.76        68

    accuracy                           0.83       179
   macro avg       0.82      0.80      0.81       179
weighted avg       0.83      0.83      0.82       179

[[100  11]
 [ 20  48]]


> Możemy zauważyć, że w przypadku drzewa decyzyjnego o maksymalnej glębokości 3 otrzymaliśmy o wiele lepszy wynik. Czy da się jeszcze lepiej? 

## Support Vector Machines (SVM)

W przypadku tego modelu warto jest przeskalować dane, ponieważ SVM jest wrażliwy na różnice w skali cech. Sprawdźmy jak to wygląda w przypadku naszego zbioru danych.

- SVM bez skalowania danych

In [581]:
from sklearn.svm import SVC

svm_unscaled_model = SVC(random_state=42)

svm_unscaled_model.fit(X_train, y_train)

evaluate_model(y_test, svm_unscaled_model)

              precision    recall  f1-score   support

           0       0.68      0.87      0.76       111
           1       0.61      0.32      0.42        68

    accuracy                           0.66       179
   macro avg       0.64      0.60      0.59       179
weighted avg       0.65      0.66      0.63       179

[[97 14]
 [46 22]]


- SVM ze skalowaniem danych

In [582]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('scaler', RobustScaler(), ['Age', 'Fare'])
    ],
    remainder='passthrough'
)

svm_scaled_model = make_pipeline(preprocessor, SVC(random_state=42))

svm_scaled_model.fit(X_train, y_train)

evaluate_model(y_test, svm_scaled_model)

              precision    recall  f1-score   support

           0       0.84      0.90      0.87       111
           1       0.82      0.72      0.77        68

    accuracy                           0.83       179
   macro avg       0.83      0.81      0.82       179
weighted avg       0.83      0.83      0.83       179

[[100  11]
 [ 19  49]]


> Możemy zauważyć, że przeskalowanie danych ma znaczący wpływ na precyzję pewnych modeli, jak i oczekiwaliśmy. Precyzja po przeskalowaniu wzrosła z `0.61` do `0.83`, co wskazuje, że dla pewnych modeli ma bardzo ważne znaczenie.

> Nie warto zapominać o tym, że zbiór testowy to tylko 20% pasażerów wybranych losowo, czyli przez przypadek mogliśmy wybrać taki zbiór danych, że model działa lepiej lub gorzej za każdym razem.

## Sprawdzenie, czy wyniki zależą od podziału danych na testowe i treningowe

In [583]:
from sklearn.model_selection import cross_val_score

def evaluate_model_cv(model, X, y, model_name):
    results = cross_val_score(model, X, y, cv=5, scoring='precision')

    print(f"Model name: {model_name}")
    print(results)
    print(f"mean: {results.mean():.3f}  std: {results.std():.3f}\n")

for model_name, model in [
    ("Dummy", dummy_model), 
    ("Tree", tree_model), 
    ("SVM trained on unscaled data", svm_unscaled_model), 
    ("SVM trained on scaled data", svm_scaled_model)
    ]:
    evaluate_model_cv(model, X, y, model_name)

Model name: Dummy
[0.4691358  0.37209302 0.37113402 0.44444444 0.41463415]
mean: 0.414  std: 0.039

Model name: Tree
[0.76056338 0.74647887 0.72727273 0.85365854 0.73913043]
mean: 0.765  std: 0.045

Model name: SVM trained on unscaled data
[0.4        0.7027027  0.76923077 0.69230769 0.72413793]
mean: 0.658  std: 0.132

Model name: SVM trained on scaled data
[0.76388889 0.76119403 0.73529412 0.81132075 0.81538462]
mean: 0.777  std: 0.031



> Możemy zauważyć, że niezależnie od wyboru danych testowych/treningowych precyzja modelu się zmienia, ale nieznacznie (odchylenie jest barzo małe). 

> Natomiast w przypadku modelu SVM(SVC) możemy zauważyć to, co już było powiedziane wcześniej: wyniki niektórych modeli bardzo zależą od przeskalowania danych. Na to wskazuje duże odchylenie standardowe (std=0.132) dla tego modelu, wytrenowanego na nieprzeskalowanych danych.

## Testowanie dla różnych hiperparametrów

**1. DecisionTreeClassifier**

In [584]:
from sklearn.model_selection import GridSearchCV

param_grid_tree = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'criterion': ['gini', 'entropy']
}

tree_search = GridSearchCV(
    DecisionTreeClassifier(random_state=5),
    param_grid_tree,
    cv=5,
    scoring='precision'
)
tree_search.fit(X_train, y_train)

def print_grid_search_results(search):
    print("Best params:\t", search.best_params_)
    print("Best score:\t", round(search.best_score_, 3))
    print("Score:\t", round(search.score(X_test, y_test), 3))

print_grid_search_results(tree_search)

Best params:	 {'criterion': 'entropy', 'max_depth': 10, 'min_samples_split': 5}
Best score:	 0.778
Score:	 0.803


In [585]:
table = pd.DataFrame(tree_search.cv_results_)
table = table[['param_max_depth', 'param_min_samples_split', 'param_criterion',
                 'mean_test_score', 'std_test_score']]
table.sort_values('mean_test_score', ascending=False).head(10)

,param_max_depth,param_min_samples_split,param_criterion,mean_test_score,std_test_score
19,10,5,entropy,0.778137,0.065915
20,10,10,entropy,0.769305,0.045576
16,5,5,entropy,0.768819,0.049961
15,5,2,entropy,0.768819,0.049961
18,10,2,entropy,0.768238,0.063050
17,5,10,entropy,0.763330,0.052644
5,5,10,gini,0.761006,0.054679
4,5,5,gini,0.753841,0.065029
3,5,2,gini,0.750067,0.066494
2,3,10,gini,0.746762,0.079822


> Analizując wyniki optymalizacji hiperparametrów, widzimy, że dla naszego zbioru danych drzewo decyzyjne osiąga lepsze rezultaty przy kryterium opartym na `entropii` oraz `większej głębokości (max_depth = 10)` w porównaniu do początkowych ustawień. Ponadto stosunkowo niskie odchylenie standardowe świadczy o tym, że model jest stabilny i wykazuje powtarzalną skuteczność pomiędzy poszczególnymi podziałami w walidacji krzyżowej

**2. SVM**

In [586]:
param_grid_svc = {
    'svc__C': [1, 10, 100],
    'svc__kernel': ['linear', 'rbf'],
    'svc__gamma': ['scale', 0.1, 1]
}

svm_scaled_model = make_pipeline(preprocessor, SVC(random_state=42))

svc_search = GridSearchCV(
    svm_scaled_model,
    param_grid_svc,
    cv=5,
    scoring='precision'
)
svc_search.fit(X_train, y_train)

print_grid_search_results(svc_search)

Best params:	 {'svc__C': 10, 'svc__gamma': 0.1, 'svc__kernel': 'rbf'}
Best score:	 0.79
Score:	 0.794


In [587]:
svc_table = pd.DataFrame(svc_search.cv_results_)
svc_table = svc_table[['param_svc__C', 'param_svc__kernel', 'param_svc__gamma',
                 'mean_test_score', 'std_test_score']]
svc_table.sort_values('mean_test_score', ascending=False).head(10)

,param_svc__C,param_svc__kernel,param_svc__gamma,mean_test_score,std_test_score
9,10,rbf,0.1,0.789737,0.079594
7,10,rbf,scale,0.785864,0.064582
13,100,rbf,scale,0.779840,0.076625
1,1,rbf,scale,0.758753,0.080059
3,1,rbf,0.1,0.756773,0.071574
15,100,rbf,0.1,0.749933,0.061771
0,1,linear,scale,0.737425,0.071191
4,1,linear,1,0.737425,0.071191
6,10,linear,scale,0.737425,0.071191
2,1,linear,0.1,0.737425,0.071191


> W czołowej szóstce tabeli znajdują się wyłącznie modele z jądrem `rbf`. Oznacza to, że granica decyzyjna dla danych z Tytanika nie jest prostą linią - relacje między cechami są nieliniowe, a model nieliniowy (rbf) radzi sobie z nimi znacznie lepiej niż płaszczyzny liniowe. 

> Podobnie do drzewa decyzyjnego, odchylenie standardowe jest stosunkowo niskie, co też świadczy, że wybrany model jest stabilny

> Po wyborze optymalnych parametrów dla obu modeli widzimy, że model `SVC` działa z precyzją nieznacznie wyższą niż `drzewo decyzyjne`. Jednak uwzględniając odchylenie standardowe, możemy stwierdzić, iż ostateczne wyniki są bardzo zbliżone.

## Wnioski

Podsumowując całą pracę nad modelami dla Tytanika, można wyciągnąć kilka ciekawych wniosków. Przede wszystkim okazało się, że skalowanie danych miało spore znaczenie - użycie `RobustScaler` na kolumnach `Fare` i `Age` było kluczowe dla modeli opartych na odległościach, takich jak `SVC`. Z kolei drzewa decyzyjne poradziły sobie z tym zupełnie naturalnie, bo skala danych nie ma na nich wpływu.

Zarówno zoptymalizowane `drzewo decyzyjne`, jak i `SVC` z jądrem rbf pokazały, że relacje w danych są nieliniowe, a ich precyzja ustabilizowała się w okolicach 77-79%, dzięki czemu modele skutecznie ograniczają liczbę fałszywych wskazań. Po optymalizacji parametrów za pomocą `GridSearchCV` model SVC działał minimalnie lepiej niż drzewo, ale w praktyce oba modele dają bardzo zbliżoną, stabilną precyzję.

Warto też dodać, że porównanie z modelem bazowym udowodniło, iż nasze algorytmy faktycznie wyciągnęły sensowne wzorce z danych, a nie działały po prostu losowo. Niskie odchylenie standardowe w walidacji krzyżowej pokazuje dodatkowo, że modele są powtarzalne i dobrze radzą sobie z różnymi podziałami danych, zamiast polegać na całkowitej losowości.